# Missing Value Analysis & Handling (GED Sample Tables)

This notebook performs missing-value analysis for the sample candidate and test tables, applies table-specific handling rules, and writes cleaned outputs plus a missing-value report.

In [6]:
from pathlib import Path
import pandas as pd

In [7]:
# ---------- Configuration ----------
project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent

candidate_file = project_root / 'data/raw_sample/candidate_table_raw_sample.xlsx'
test_file = project_root / 'data/raw_sample/test_data_table_raw_sample.xlsx'
output_dir = project_root / 'data/processed'
output_dir.mkdir(parents=True, exist_ok=True)

candidate_output = output_dir / f'candidate_cleaned{candidate_file.suffix}'
test_output = output_dir / f'test_cleaned{test_file.suffix}'
report_output = output_dir / 'missing_value_report.csv'

candidate_file, test_file, output_dir

(PosixPath('/Users/dagur/msba/GED_EDA_Project/data/raw_sample/candidate_table_raw_sample.xlsx'),
 PosixPath('/Users/dagur/msba/GED_EDA_Project/data/raw_sample/test_data_table_raw_sample.xlsx'),
 PosixPath('/Users/dagur/msba/GED_EDA_Project/data/processed'))

In [8]:
# ---------- I/O helpers ----------
def read_table(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == '.csv':
        return pd.read_csv(path)
    if suffix in {'.xlsx', '.xls'}:
        return pd.read_excel(path)
    if suffix == '.parquet':
        return pd.read_parquet(path)
    raise ValueError(f'Unsupported file format: {path}')

def write_table(df: pd.DataFrame, path: Path) -> None:
    suffix = path.suffix.lower()
    if suffix == '.csv':
        df.to_csv(path, index=False)
        return
    if suffix in {'.xlsx', '.xls'}:
        df.to_excel(path, index=False)
        return
    if suffix == '.parquet':
        df.to_parquet(path, index=False)
        return
    raise ValueError(f'Unsupported output format: {path}')

def missing_report(df: pd.DataFrame, table_name: str) -> pd.DataFrame:
    missing_count = df.isna().sum()
    report = pd.DataFrame({
        'table': table_name,
        'column': missing_count.index,
        'missing_count': missing_count.values,
    })
    report['missing_pct'] = (report['missing_count'] / len(df) * 100).round(2)
    return report.sort_values('missing_count', ascending=False)

In [9]:
# ---------- Cleaning rules ----------
CANDIDATE_CATEGORICAL_FILL = {
    'LAST_YEAR_INCOME': 'UNKNOWN',
    'PREP_CENTER_STATE': 'UNKNOWN',
    'ENROLLMENT_STATUS': 'UNKNOWN',
    'PREP_CENTER': 'UNKNOWN',
    'SCHOOL_INCOMPLETE_REASON': 'UNKNOWN',
    'TESTING_REASON': 'UNKNOWN',
    'GENDER': 'UNKNOWN',
    'ETHNICITY': 'UNKNOWN',
    'HIGHEST_GRADE_COMPLETED': 'UNKNOWN',
    'C_COUNTY': 'UNKNOWN',
    'C_STATE': 'UNKNOWN',
    'ZIP': 'UNKNOWN',
}

TEST_CATEGORICAL_FILL = {
    'T_STATE': 'UNKNOWN',
    'T_POSTAL_CODE': 'UNKNOWN',
    'T_NAME': 'UNKNOWN',
    'T_ADDRESS': 'UNKNOWN',
    'TEST_CENTER_ID': 'UNKNOWN',
    'T_CITY': 'UNKNOWN',
    'EXAM_SUBJECT': 'UNKNOWN',
}

TEST_BINARY_FILL_ZERO = ['FIRST_COMPLETE', 'FIRST_PASSED', 'MAKE_COMPLETE', 'MAKE_PASSED']

def clean_candidate_table(df: pd.DataFrame) -> pd.DataFrame:
    cleaned = df.copy()
    for col, fill_value in CANDIDATE_CATEGORICAL_FILL.items():
        if col in cleaned.columns:
            cleaned[col] = cleaned[col].astype('string').fillna(fill_value)

    if 'STUDIED_FOR_GED' in cleaned.columns:
        mode = cleaned['STUDIED_FOR_GED'].mode(dropna=True)
        fill_value = mode.iloc[0] if not mode.empty else 0
        cleaned['STUDIED_FOR_GED'] = cleaned['STUDIED_FOR_GED'].fillna(fill_value)

    if 'CREDENTIAL_DATE' in cleaned.columns:
        cleaned['CREDENTIAL_DATE_MISSING'] = cleaned['CREDENTIAL_DATE'].isna().astype('int8')
    return cleaned

def clean_test_table(df: pd.DataFrame) -> pd.DataFrame:
    cleaned = df.copy()
    for col, fill_value in TEST_CATEGORICAL_FILL.items():
        if col in cleaned.columns:
            cleaned[col] = cleaned[col].astype('string').fillna(fill_value)

    for col in TEST_BINARY_FILL_ZERO:
        if col in cleaned.columns:
            cleaned[col] = cleaned[col].fillna(0)

    for col in ['RESULT_ID', 'SCORE', 'EXAM_START']:
        if col in cleaned.columns:
            cleaned[f'{col}_MISSING'] = cleaned[col].isna().astype('int8')
    return cleaned

In [10]:
# ---------- Run analysis ----------
candidate_df = read_table(candidate_file)
test_df = read_table(test_file)

candidate_report = missing_report(candidate_df, 'candidate')
test_report = missing_report(test_df, 'test')

display(candidate_report[candidate_report['missing_count'] > 0].head(20))
display(test_report[test_report['missing_count'] > 0].head(20))

combined_report = pd.concat([candidate_report, test_report], ignore_index=True)
combined_report.to_csv(report_output, index=False)
report_output

,table,column,missing_count,missing_pct
9,candidate,LAST_YEAR_INCOME,4284,85.68
39,candidate,CREDENTIAL_DATE,3421,68.42
37,candidate,PREP_CENTER_STATE,3194,63.88
36,candidate,PREP_CENTER,3185,63.70
38,candidate,ENROLLMENT_STATUS,3185,63.70
15,candidate,SCHOOL_INCOMPLETE_REASON,1066,21.32
14,candidate,TESTING_REASON,1045,20.90
8,candidate,GENDER,1036,20.72
35,candidate,ETHNICITY,1036,20.72
16,candidate,STUDIED_FOR_GED,1032,20.64


,table,column,missing_count,missing_pct
9,test,FIRST_PASSED,16563,59.51
8,test,FIRST_COMPLETE,16563,59.51
11,test,MAKE_PASSED,16563,59.51
10,test,MAKE_COMPLETE,16563,59.51
6,test,EXAM_START,1632,5.86
5,test,RESULT_ID,642,2.31
7,test,SCORE,642,2.31
14,test,T_STATE,141,0.51
17,test,T_POSTAL_CODE,85,0.31
13,test,T_NAME,12,0.04


PosixPath('/Users/dagur/msba/GED_EDA_Project/data/processed/missing_value_report.csv')

In [11]:
# ---------- Clean and save ----------
cleaned_candidate = clean_candidate_table(candidate_df)
cleaned_test = clean_test_table(test_df)

write_table(cleaned_candidate, candidate_output)
write_table(cleaned_test, test_output)

candidate_output, test_output

(PosixPath('/Users/dagur/msba/GED_EDA_Project/data/processed/candidate_cleaned.xlsx'),
 PosixPath('/Users/dagur/msba/GED_EDA_Project/data/processed/test_cleaned.xlsx'))

## For 3M+ rows
For production-scale data, use CSV or Parquet (instead of Excel) for faster read/write and lower memory pressure.